# QLoRA vs. LoRA vs. full fine-tuning of BERT

Trains `bert-base-uncased` for IMDB sentiment three ways on the same GPU and split: QLoRA (frozen 4-bit base + LoRA adapters), LoRA (frozen 32-bit base + the same adapters) and full fine-tuning, then compares accuracy, peak GPU memory, training time and latency. LoRA vs. QLoRA isolates the effect of quantization; LoRA vs. full isolates the effect of training only adapters.

All logic lives in `src/qlora_bert/`. **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Install

In [ ]:
%cd /content
!rm -rf qlora-bert-sentiment-classification
!git clone -q -b polish/measured-comparison https://github.com/dojedaro/qlora-bert-sentiment-classification.git
%cd qlora-bert-sentiment-classification
!pip install -q -e ".[gpu]"

## Train three ways
Each method runs in its own process, so peak memory is measured from a clean GPU. About 10–15 minutes each on a T4.

In [ ]:
!python -m qlora_bert.experiment --method qlora --out results

In [ ]:
!python -m qlora_bert.experiment --method lora --out results

In [ ]:
!python -m qlora_bert.experiment --method full --out results

## Compare

In [ ]:
import json

import pandas as pd

rows = {}
for method in ("qlora", "lora", "full"):
    r = json.load(open(f"results/{method}.json"))
    rows[method] = {
        **{f"test_{k}": v for k, v in r["test"].items()},
        "validation_accuracy": r["validation"]["accuracy"],
        "trainable_parameters": r["trainable_parameters"],
        "peak_gpu_memory_gb": r["peak_gpu_memory_gb"],
        "train_minutes": round(r["train_seconds"] / 60, 1),
        "latency_ms_batch1": r["latency_ms_batch1"],
    }
pd.DataFrame(rows)

## Download the results

In [ ]:
from google.colab import files

for method in ("qlora", "lora", "full"):
    files.download(f"results/{method}.json")